# 17B — Cycle 24 → Cycle 25 multimodal data-readiness audit

## Stage 1: metadata content, identities and catalogue scope

**Runnable implementation, not an executed result for the user's archive.** This first stage performs bounded exact-object downloads of five CSVs, records object generations/checksums, and streams the files into a local SQLite audit database. It does not require pandas, torch or CUDA.

The final line `STAGE1_COMPLETE_REVIEW_REQUIRED` means the program finished; it does **not** mark the full 17B audit complete or authorise training. Official file contents, labels and cloud resources are never changed.


In [ ]:
from pathlib import Path
import json
import sys

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents]
             if (p / "configs/aia17_metadata_stage1.json").is_file()),
            Path.home() / "solar_flare_aia")
if not (ROOT / "src/aia17_metadata_audit.py").is_file():
    raise FileNotFoundError("Install the complete AIA 17A/17B package before running this notebook.")
sys.path.insert(0, str(ROOT / "src"))
from aia17_metadata_audit import check_protocol, run_pipeline
CONFIG = json.loads((ROOT / "configs/aia17_metadata_stage1.json").read_text())
check_protocol(CONFIG)
print("ROOT:", ROOT)
print("Protocol status:", CONFIG["split_status"])
print("Training authorised:", CONFIG["training_authorised"])


## 1. Explicit scope and download guardrails

Five sources: established baseline manifest; full curated targets; extension targets; raw 96-minute SHARP history; HEK M/X event catalogue. **No full twelve-minute table, images or model checkpoints are downloaded.** Total source-size limit: 450 MiB, with at least 1 GiB additional free disk reserved. This cap bounds source payload size, **not a monetary spending limit**; API requests, retries and transfers remain subject to Cloud Storage charges.

The sources are pinned on first execution. Later runs reuse and verify those versions instead of silently switching to newer data. Content-encoded or unexpectedly large files cause an explicit stop. The default workspace is `~/aia17_metadata_stage1`, separate from Git.


In [ ]:
WORK = Path.home() / "aia17_metadata_stage1"
print("Local audit workspace:", WORK)
print("CSV source-size cap (MiB):", CONFIG["max_total_source_bytes"] / 1024**2)
print("Source count:", len(CONFIG["sources"]))


## 2. Stage and inspect the metadata

Running the next cell downloads any uncached source versions and performs the audit. Run it once; do not run a second terminal copy concurrently. Authentication uses the existing local gcloud login, never a pasted key.

The terminal launcher runs this same helper code, so a notebook server is not needed for the initial Cloud Shell run.


In [ ]:
OUT = run_pipeline(ROOT, WORK)
print("Report directory:", OUT)


## 3. Read the actual report

The report covers full-source row/year counts, validity of official-label values, duplicate IDs and conflicting labels, extension/master and baseline/master identity comparisons, available SHARP columns and numeric missingness, raw timestamp representations, raw 96-minute cadence gaps, and event class/time-order checks.

Annual counts describe **unfiltered source rows**, not image-object completeness. No label repair, imputation, source concatenation or timescale assignment is performed.


In [ ]:
REPORT = json.loads((OUT / "stage1_report.json").read_text())
print((OUT / "stage1_summary.md").read_text())


## 4. Remaining gates: not implemented by this stage

AIA object/manifest reconciliation across both storage roots; source tensor/actual observation-time integrity; qualified AIA/SHARP 6h/12h/24h sequence coverage; timescale and reporting-latency conversion; frozen split/AR/purge policy; independent label/follow-up coverage including right-censoring; a complete past-only GOES feature table; optional HMI coverage; checkpoint compatibility/loadability.

Later 17B stages must resolve these before 17C. The `label_48h_final` column in a source is evidence of its schema, **not proof of every label's scientific correctness**. Catalogue maximum event time is not a certified coverage endpoint.

## 5. Preserve outputs

Keep `stage1_summary.md`, `stage1_report.json`, the CSV summaries and source-lock/config snapshots. The SQLite database and cached raw CSVs remain outside Git. Review the small outputs before a coherent batch commit; do not `git add .` indiscriminately. This notebook performs no Git writes.


In [ ]:
print("Stage:", REPORT["stage"])
print("Status:", REPORT["status"])
print("Training authorised:", REPORT["training_authorised"])
for gate in REPORT["remaining_training_gates"]:
    print("PENDING:", gate)


## Additional readiness requirements for uncertainty and calibration

The adopted standard is **Performance → Calibration → Uncertainty → Robustness → Explainability → Statistical significance**. Read `docs/AIA_UQ_CALIBRATION_PROTOCOL.md`.

Subsequent 17B stages must determine AR/track grouping, independent positive-group support, separated calibration/development support, year/regime coverage, time/quality/missingness and matched sample sets for paired comparisons. Stage 17C must later verify score export and any stochastic inference mechanism.

**The existing stage-1 computation has not been extended to run these checks or estimate uncertainty.** Its completion status remains `STAGE1_COMPLETE_REVIEW_REQUIRED`, not training authorisation. Finish any running stage-1 audit unchanged; do not rerun it merely to obtain this protocol addition.
